

# Data Exploration Notebook

## Purpose

This notebook documents the initial exploration of three data sources — FEMA, Census, and HUD — in support of a predictive model for federal disaster assistance. The goal is to understand the raw structure of each source: what fields are available, how county identifiers (FIPS codes) are represented, what date formats are used, and what data quality issues will need to be addressed during cleaning.

**Research Question:** Can we predict the level of federal disaster assistance allocated to a county based on its pre-disaster demographic characteristics and housing conditions?

Because we are studying pre-disaster conditions rather than tracking outcomes over time, this is a **cross-sectional study** — each observation represents a county at a point in time prior to a declared disaster, not a time series.


## Exploration Plan

### FEMA Disaster Declarations

The expected unit of analysis is one row per county per disaster event. We will:

- Inspect the top-level JSON structure, including metadata and the `DisasterDeclarationsSummaries` field, then convert it to a DataFrame.
- Identify key fields relevant to schema design, particularly columns containing FIPS codes, date information, and incident or disaster type.
- Assess missing values across all columns.

### Census API

The Census data is expected to contain demographic variables such as population, income, poverty rates, and unemployment. We will:

- Examine the first ten rows and column data types, coercing numeric fields (e.g., income, unemployment rate) to floats using `errors='coerce'` so that malformed entries are returned as `NaN` rather than causing parsing failures.
- Review summary statistics to identify outliers or implausible values (e.g., negative income figures).
- Validate FIPS code completeness by comparing the number of unique counties in the data against the known count of U.S. counties (~3,200), and flag any counties with substantial missing data.

### HUD Data

The HUD dataset contains housing cost and condition variables. We will:

- Inspect column names, data types, shape, and the first few rows. An unexpected number of columns can itself signal a structural anomaly worth investigating.
- Check for duplicate rows and determine whether any duplication is a genuine data quality issue or an artifact of the source's data entry design (e.g., multiple program records per county).



## 1. FEMA Disaster Declarations (API Exploration)


The main changes I made: converted the informal stream-of-consciousness planning notes into structured prose with clear intent statements, grouped related checks under each data source, removed redundant phrasing, and made the cross-sectional framing explicit upfront since it shapes the entire analysis design.

In [ ]:
import requests
import pandas as pd
import json
from pprint import pprint


print("FEMA API DATA EXPLORATION")


# lets first fetch just 10 records to examine structure
fema_url = "https://www.fema.gov/api/open/v2/DisasterDeclarationsSummaries"
params = {"$top": 10}

try:
    response = requests.get(fema_url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()
    
    print("\nAPI Response Structure:")
    print("Top-level keys:", list(data.keys()))
    
    if 'DisasterDeclarationsSummaries' in data:
        records = data['DisasterDeclarationsSummaries']
        print(f"\nNumber of records fetched: {len(records)}")
        
        # checking the first record in detail

        print("FIRST RECORD STRUCTURE (detailed view):")
        pprint(records[0], depth=2)
        

        # converting to DataFrame to see its structure
        df = pd.DataFrame(records)

        print("DATAFRAME OVERVIEW:")
        print(f"Shape: {df.shape}")
        print(f"\nColumns ({len(df.columns)}):")
        for col in df.columns:
            print(f"  - {col}")
        
        print("DATA TYPES:")
        print(df.dtypes)
        
        print("SAMPLE DATA (first 3 rows):")
        print(df.head(3))
        
        print("MISSING VALUES:")
        missing = df.isnull().sum()
        print(missing[missing > 0])
        
        print("KEY FIELDS FOR SCHEMA DESIGN:")
        print("\nFIPS-related fields:")
        fips_cols = [col for col in df.columns if 'fips' in col.lower()]
        print(fips_cols)
        if fips_cols:
            for col in fips_cols:
                print(f"\n{col}:")
                print(f"  Type: {df[col].dtype}")
                print(f"  Sample values: {df[col].head(3).tolist()}")
                print(f"  Null count: {df[col].isnull().sum()}")
        
        print("\nDate-related fields:")
        date_cols = [col for col in df.columns if 'date' in col.lower()]
        print(date_cols)
        if date_cols:
            for col in date_cols:
                print(f"\n{col}:")
                print(f"  Type: {df[col].dtype}")
                print(f"  Sample values: {df[col].head(3).tolist()}")
        
        print("\nIncident/Disaster type fields:")
        incident_cols = [col for col in df.columns if any(word in col.lower() for word in ['incident', 'disaster', 'type'])]
        print(incident_cols)
        if incident_cols:
            for col in incident_cols:
                print(f"\n{col}:")
                print(f"  Type: {df[col].dtype}")
                print(f"  Unique values: {df[col].nunique()}")
                print(f"  Sample values: {df[col].unique()[:5].tolist()}")
        
        # now let save this sample for reference
        df.to_csv('/Users/dcnguyen060899/Downloads/SU_Winter_2026/CSPC_5071/su_cpsc5071_group_project_sad_sravya_anushka_duy/data/fema_sample_data.csv', index=False)
        print("Sample data saved to: fema_sample_data.csv")
        
except Exception as e:
    print(f"Error: {e}")

FEMA API DATA EXPLORATION

API Response Structure:
Top-level keys: ['metadata', 'DisasterDeclarationsSummaries']

Number of records fetched: 10
FIRST RECORD STRUCTURE (detailed view):
{'declarationDate': '2024-08-09T00:00:00.000Z',
 'declarationRequestNumber': '24122',
 'declarationTitle': 'LEE FALLS FIRE',
 'declarationType': 'FM',
 'designatedArea': 'Washington (County)',
 'designatedIncidentTypes': 'R',
 'disasterCloseoutDate': None,
 'disasterNumber': 5529,
 'femaDeclarationString': 'FM-5529-OR',
 'fipsCountyCode': '067',
 'fipsStateCode': '41',
 'fyDeclared': 2024,
 'hash': 'ae87cf3c6ed795015b714af7166c7c295b2b67c7',
 'hmProgramDeclared': True,
 'iaProgramDeclared': False,
 'id': '09e3f81a-5e16-4b72-b317-1c64e0cfa59c',
 'ihProgramDeclared': False,
 'incidentBeginDate': '2024-08-08T00:00:00.000Z',
 'incidentEndDate': None,
 'incidentId': '2024081001',
 'incidentType': 'Fire',
 'lastIAFilingDate': None,
 'lastRefresh': '2024-08-27T18:22:14.800Z',
 'paProgramDeclared': True,
 'placeC

## FEMA API — Key Findings and Design Decisions

**Schema and Column Selection**

The API returns 28 columns, but a significant portion are internal metadata fields that carry no analytical value. Fields such as `hash`, `lastRefresh`, and `femaDeclarationString` will be dropped during cleaning. The core fields we intend to retain are `fipsStateCode`, `fipsCountyCode`, `disasterNumber`, `declarationDate`, the four program declaration flags (`ihProgramDeclared`, `iaProgramDeclared`, `paProgramDeclared`, `hmProgramDeclared`), and incident type information.

**FIPS Code Structure**

The FIPS code is stored as two separate fields — `fipsStateCode` (e.g., `'41'` for Oregon) and `fipsCountyCode` (e.g., `'067'` for Washington County). To produce a standard five-digit FIPS code usable as a join key across datasets, these two fields will need to be concatenated during cleaning. Importantly, neither field contains missing values in this sample, which is a promising sign for join completeness.

**Date Format**

All date fields are stored as ISO 8601 strings with a UTC timestamp suffix (e.g., `'2024-08-09T00:00:00.000Z'`). Since we only need date-level precision for a cross-sectional analysis, the timestamp component will be stripped, retaining only the `YYYY-MM-DD` portion.

**Missing Values**

Missing values are concentrated in fields that are either event-dependent (`incidentEndDate`, `disasterCloseoutDate`) or program-specific (`lastIAFilingDate`, `designatedIncidentTypes`). These are expected given the nature of disaster declarations, where not all events have a defined end date or closeout at the time of record creation. They do not raise immediate concern for the core analysis.

## 2. Census ACS 5-Year Estimates (API Exploration)

In [ ]:

print("PART 2: CENSUS ACS 5-YEAR ESTIMATES API EXPLORATION")

# Census API requires a key, but you can get one for free at:
# https://api.census.gov/data/key_signup.html
# For exploration, we'll use the public endpoint which has rate limits

census_base_url = "https://api.census.gov/data/2022/acs/acs5"

# check the document in the data folder to understand demographic variables we're exploring:
# B01001_001E - Total Population
# B19013_001E - Median Household Income
# B17001_002E - Income below poverty level
# B23025_005E - Unemployment

variables = {
    'B01001_001E': 'total_population',
    'B19013_001E': 'median_household_income', 
    'B17001_002E': 'poverty_count',
    'B23025_005E': 'unemployment_count',
    'B23025_003E': 'labor_force_count'
}

# Request format: get=NAME,B01001_001E&for=county:*&in=state:*
params = {
    'get': 'NAME,' + ','.join(variables.keys()),
    'for': 'county:*',
    'in': 'state:*'
}

try:
    print("\nFetching Census ACS data...")
    print("Note: This may take 30-60 seconds due to the large dataset")
    
    response = requests.get(census_base_url, params=params, timeout=60)
    response.raise_for_status()
    
    data = response.json()
    
    # the first row is headers
    headers = data[0]
    records = data[1:]
    
    print(f"\nFetched {len(records)} county records")
    print(f"Columns: {headers}")
    
    # converting to DataFrame
    df_census = pd.DataFrame(records, columns=headers)
    
    print("CENSUS DATA STRUCTURE:")
    print(df_census.head(10))
    
    print("DATA TYPES (before conversion):")
    print(df_census.dtypes)
    
    # converting numeric columns
    for var_code in variables.keys():
        if var_code in df_census.columns:
            df_census[var_code] = pd.to_numeric(df_census[var_code], errors='coerce')
    
    print("FIPS CODE CONSTRUCTION:")
    df_census['state_fips'] = df_census['state'].str.zfill(2)
    df_census['county_fips'] = df_census['county'].str.zfill(3)
    df_census['fips'] = df_census['state_fips'] + df_census['county_fips']
    
    print(df_census[['NAME', 'state', 'county', 'state_fips', 'county_fips', 'fips']].head(10))
    
    print(f"\nTotal unique FIPS codes: {df_census['fips'].nunique()}")
    
    print("VARIABLE ANALYSIS:")
    
    for var_code, var_name in variables.items():
        if var_code in df_census.columns:
            print(f"\n{var_name} ({var_code}):")
            print(f"  Data type: {df_census[var_code].dtype}")
            print(f"  Null/Missing values: {df_census[var_code].isnull().sum()}")
            print(f"  Min: {df_census[var_code].min()}")
            print(f"  Max: {df_census[var_code].max()}")
            print(f"  Mean: {df_census[var_code].mean():.2f}")
            
            # checking for special values (the documentation notes that census uses -666666666 for missing)
            negative_count = (df_census[var_code] < 0).sum()
            if negative_count > 0:
                print(f"  WARNING: {negative_count} negative values detected (may indicate missing/suppressed data)")
    
    print("DERIVED CALCULATIONS:")
    
    # Calculate poverty rate
    df_census['poverty_rate'] = (df_census['B17001_002E'] / df_census['B01001_001E'] * 100)
    
    # Calculate unemployment rate
    df_census['unemployment_rate'] = (df_census['B23025_005E'] / df_census['B23025_003E'] * 100)
    
    print("Poverty Rate Statistics:")
    print(df_census['poverty_rate'].describe())
    
    print("\nUnemployment Rate Statistics:")
    print(df_census['unemployment_rate'].describe())
    
    # Save sample
    df_census_sample = df_census.head(100)
    df_census_sample.to_csv('/Users/dcnguyen060899/Downloads/SU_Winter_2026/CSPC_5071/su_cpsc5071_group_project_sad_sravya_anushka_duy/data/census_exploration.csv', index=False)
    print("\n✓ Saved sample to: census_exploration.csv")
    
    # storing
    print(f"\nTotal counties in Census data: {len(df_census)}")
    
except Exception as e:
    print(f"Error exploring Census API: {e}")
    import traceback
    traceback.print_exc()
    print("\nNote: Census API may require an API key for high-volume requests.")
    print("Get free key at: https://api.census.gov/data/key_signup.html")

PART 2: CENSUS ACS 5-YEAR ESTIMATES API EXPLORATION

Fetching Census ACS data...
Note: This may take 30-60 seconds due to the large dataset

Fetched 3222 county records
Columns: ['NAME', 'B01001_001E', 'B19013_001E', 'B17001_002E', 'B23025_005E', 'B23025_003E', 'state', 'county']
CENSUS DATA STRUCTURE:
                       NAME B01001_001E B19013_001E B17001_002E B23025_005E  \
0   Autauga County, Alabama       58761       68315        6630         752   
1   Baldwin County, Alabama      233420       71039       23445        3825   
2   Barbour County, Alabama       24877       39712        5280         516   
3      Bibb County, Alabama       22251       50669        4297         786   
4    Blount County, Alabama       59077       57440        8277        1578   
5   Bullock County, Alabama       10328       36136        2687         144   
6    Butler County, Alabama       18981       44429        3937         548   
7   Calhoun County, Alabama      116162       54339       20267 



## Key Observations from Census ACS Exploration

**Coverage and Scale**

The API returned 3,222 county-level records, which aligns closely with the known count of U.S. counties and county-equivalents (~3,143 counties plus additional territories and independent cities). This gives us confidence that the dataset provides near-complete national coverage with no obvious systematic under-representation.

**Variable Naming**

All demographic variables are returned under Census API codes (e.g., `B01001_001E`, `B19013_001E`) rather than descriptive names. These must be renamed to human-readable labels — such as `total_population`, `median_household_income`, and `poverty_count` — before the data is usable in analysis or shared with stakeholders.

**Data Types**

All columns, including numeric fields, are initially parsed as `object` type. Numeric variables must be explicitly cast to `int64` or `float64` to support arithmetic operations. The `errors='coerce'` parameter should be applied during conversion so that any non-numeric entries are silently replaced with `NaN` rather than raising parsing errors.

**Sentinel Value for Missing Income**

The minimum value for `median_household_income` is `-666,666,666`, which is a documented Census Bureau sentinel value indicating suppressed or unavailable data — not a real income figure. This value must be replaced with `NaN` during cleaning, for example via `df.replace(-666666666, np.nan, inplace=True)`, before any summary statistics or model inputs are computed from this column.

**FIPS Code Construction**

The Census API returns the state and county FIPS components in separate columns (`state` and `county`), which are concatenated to form the standard five-digit county FIPS code (e.g., `'01'` + `'001'` = `'01001'`). This matches the approach required to join with FEMA data, confirming a consistent key structure across both sources.

**Derived Variable Validation**

The poverty and unemployment rates computed from the raw counts fall within plausible ranges for U.S. counties. The mean poverty rate of 14.6% is consistent with national averages, and the high standard deviation of 7.5 percentage points reflects the well-documented disparity between affluent suburban counties and high-poverty rural areas — such as counties in Native American reservations or Appalachian regions, where the maximum reaches 65.6%. Crucially, no county returns a poverty rate below zero or above 100%, confirming that the underlying count fields are internally consistent and the derived rates are arithmetically valid.


## 3. HUD Fair Market Rents (CSV Exploration)

In [ ]:
import pandas as pd

df_hud = pd.read_csv('/Users/dcnguyen060899/Downloads/SU_Winter_2026/CSPC_5071/su_cpsc5071_group_project_sad_sravya_anushka_duy/data/FY25_FMRs-Table 1.csv')

print("Shape:", df_hud.shape)
print("\nColumns:")
print(df_hud.columns.tolist())
print("\nData types:")
print(df_hud.dtypes)
print("\nFirst 10 rows:")
print(df_hud.head(10))
print("\nSample FIPS-related columns:")
fips_cols = [col for col in df_hud.columns if 'fips' in col.lower() or 'code' in col.lower()]
print(df_hud[fips_cols].head(10) if fips_cols else "No FIPS columns found by name")

# remember the duplicate earlier, so let check for the New England subdivision rows
print(f"\nRows with county_town_name filled (NE subdivisions): {df_hud['county_town_name'].notna().sum()}")

# take a look at 5-digit county FIPS and check for duplicates
df_hud['county_fips'] = df_hud['fips'].apply(lambda x: str(x).zfill(10)[:5])
print(f"Total rows: {len(df_hud)}")
print(f"Unique 5-digit county FIPS: {df_hud['county_fips'].nunique()}")
print(f"Duplicate FIPS count: {len(df_hud) - df_hud['county_fips'].nunique()}")

# print out an example of duplicates so we can get a sense what it looks like to have multiple town within a single county
duplicated_fips = df_hud[df_hud['county_fips'].duplicated(keep=False)].sort_values('county_fips')
print("\nExample of duplicate county FIPS:")
print(duplicated_fips[['stusps', 'countyname', 'county_town_name', 'county_fips', 'fmr_2']].head(10))


Shape: (4764, 14)

Columns:
['stusps', 'state', 'hud_area_code', 'countyname', 'county_town_name', 'metro', 'hud_area_name', 'fips', 'pop2022', 'fmr_0', 'fmr_1', 'fmr_2', 'fmr_3', 'fmr_4']

Data types:
stusps              object
state                int64
hud_area_code       object
countyname          object
county_town_name    object
metro                int64
hud_area_name       object
fips                 int64
pop2022              int64
fmr_0                int64
fmr_1                int64
fmr_2                int64
fmr_3                int64
fmr_4                int64
dtype: object

First 10 rows:
  stusps  state     hud_area_code       countyname county_town_name  metro  \
0     AL      1  METRO33860M33860   Autauga County              NaN      1   
1     AL      1  METRO19300M19300   Baldwin County              NaN      1   
2     AL      1  NCNTY01005N01005   Barbour County              NaN      0   
3     AL      1  METRO13820M13820      Bibb County              NaN      1   



## Key Observations from HUD Fair Market Rent Exploration

**Row Count and Initial Anomaly**

The dataset contains 4,764 rows, substantially more than the ~3,243 counties in the United States. This immediately signals that the dataset does not follow a simple one-row-per-county structure and warrants further investigation.

**Source of Duplication: Town-Level Subdivisions**

The excess rows are explained by 1,605 entries where `county_town_name` is populated, indicating that HUD publishes separate Fair Market Rent (FMR) figures for individual towns or subdivisions within certain counties, most notably in New England states like Connecticut. This is by design: HUD sets FMR at the metropolitan area or subdivision level, not strictly at the county level. As a result, 1,536 FIPS codes appear more than once in the dataset, even though there are only 3,228 unique five-digit county FIPS values.

**Implication for Schema Design**

This duplication creates a primary key violation for any schema that expects one row per county. For example, Fairfield County, CT (`FIPS 09001`) appears across multiple town-level entries with meaningfully different `fmr_2` values ranging from $1,967 to $2,610. Since our analysis operates at the county level, we cannot arbitrarily select any single town's FMR as representative.

**Proposed Resolution**

The most principled approach is to aggregate town-level FMR values up to the county level using a population-weighted average, which ensures that larger towns exert proportionally greater influence on the county estimate. This will produce a single, defensible FMR figure per county FIPS code and restore the one-to-one key relationship required for joining with the FEMA and Census datasets.


## Problem Arises:

Below are the problems we summary from the intial observation of the shape, data type, statistics, missing values etc from the three data sources:

### First problems:
Looking at the actual FIPS codes that returned by all three datasets, we first noticed that FEMA provided two seperate string fields. What we have is for example: fipsStateCode: '41' and fipsCountyCode: '067, but we need fips: '41067'. The Census Bureau also return something similar, seperate state and county fields as strings. Now for HUD, we see that it presented a nine digit integer where the county FIPS code occupied only the first five positions. So the remaining four digits encoding sub-county geographic information.

Immediately, we know that we can't joining wasn't straight forward. So we discussed, our inital reasoning process to tackle this problem is to converted all FIPS codes to integer, however, this would have dropped leading zeros from codes like '01001' for Autauga County, Alabama. '1001 here is the state information we would have lost. Also, we could have converted all codes to nine-digit strings by padding, but this would have created artificial complexity when the trailing four digits carried no meaning for county-level analysis. The optimal solution standardized all codes to five-digit zero-padded strings, which preserved the complete county identifier while discarding HUD's sub-county extensions that were irrelevant to our analytical grain.

### Second Problems:
When we look at the HUD dataset, it has 4763 rows for a counntry cobntaining around 3200 counties, implying that many geographic areas appeared multiple times. So we had investigated and realized that some records represented metropolitan statistical areas spanning multiple counties, while others represented town-level subdivisions within New England counties. So what this mean is that this structure created a one to many relationship problem where a single county FIPS code could match multiple HUD records.

So that mean if we do join, if there is 5 row for fips representing say Washington County and only two disasters for fact table (FEMA), then we would expect only two disaster in total. But if we proceed with join, then we get 10 disaster which is not true. So we could employed population weighted averaging, which combined multiple sub-county FMR values into a single conuty-level representative value where each area's contribution reflected its population share.

### Third Problems:
Another problem we face might seem straight forward, where the variable 19013_001E for median household income contained the value negative 666,666,666 for one county. We checked the data entry documentation and apparently this value indicates a missing value. So our solution converted sentinel values to explicit nulls, which database systems and statistical software handle appropriately. Null values are excluded from aggregation functions like average and sum, preventing them from distorting calculations.

### Fourth Problems:
Notice that FEMA dataset provided events from 2020 through 2024, Census represented 2018 through 2022 aggregated to a 2022 snapshot, and lastly HUD reflected fiscal year 2025 housing costs. If we look at this, our initial concern were that joining data from different years violated temporal consistency requirements for predictive modeling. However, as we research more into our problem, and given Duy economics background, we realized that there is a different between time series and cross sectional analysis with temporal context. Our research question in facts employed the cross sectional framework, treating county demographics and housing costs as relatively stable characteristics that explain variation in disaster frequency across counties over a multi year period. By acknowledge this, we came up with the solution that we were not measuring year-specific conditions but rather using point-in-time snapshots as proxies for persistent county characteristics. This framing transformed the temporal misalignment from a problem into an accepted limitation with clear analytical implications.

### Fifth Problems:
Lastly, when we try to look for methodology and syntax to collect data through FEMA API, it returned exactly 1000 records despite our filter requesting all disasters from 2020 forward. Basically this hard limit from the API, we are require to write a loop to multiple requests to retrieve the complete dataset. This were initially invisible during our schema design without actually trying to retrieve the actual data.

What we did were implemented something called a pagination loop such that it incremented the dollar sign skip parameter by 1000 after each request until the API returned an empty response, indicating all records had been retrieved. This algorithm allow us to understand how OData query parameters function and how to detect completion conditions in API responses.


## Implementation

Discovering these insights through our data collection and exploration, we able to directly informed every design decision in the final pipeline. Next, now that we have the information regarding understanding our data domain and structure, we can next think about what we had learned early in the class. Let pedagogically design our ER diagram and translate it into relational schemas?